In [ ]:
# ===============================================
# Fish Activity Classification using CNN (TensorFlow)
# ===============================================

# Step 1: Import Libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
import os

# Step 2: Define paths to your dataset
train_dir = 'C:\\Users\\STF8586\\OneDrive - University of Derby\\Desktop\\S.K\\EffiDataset\\train'
val_dir = 'C:\\Users\\STF8586\\OneDrive - University of Derby\\Desktop\\S.K\\EffiDataset\\val'
test_dir = 'C:\\Users\\STF8586\\OneDrive - University of Derby\\Desktop\\S.K\\EffiDataset\\test'

# Step 2b: Create output folder
output_dir = 'C:\\Users\\STF8586\\OneDrive - University of Derby\\Desktop\\S.K\\results'
os.makedirs(output_dir, exist_ok=True)

# Step 3: Image Data Generators with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Load images from folders
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

# Step 4: Build CNN Model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Binary classification
])

# Step 5: Compile Model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Step 6: Define Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint(os.path.join(output_dir, 'best_model.h5'), monitor='val_loss', save_best_only=True)

# Step 7: Train Model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop, checkpoint]
)

# Step 8: Evaluate Model on Test Set
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc*100:.2f}%")

# Step 8b: Make predictions on test set
test_generator.reset()
pred_probs = model.predict(test_generator)
pred_classes = (pred_probs > 0.5).astype(int)

# Save predictions and true labels
np.save(os.path.join(output_dir, 'test_predictions.npy'), pred_classes)
np.save(os.path.join(output_dir, 'test_true_labels.npy'), test_generator.classes)
print(f"Predictions and true labels saved in '{output_dir}' folder!")

# Step 9: Save the entire model
model.save(os.path.join(output_dir, 'fish_activity_model.h5'))
print(f"Model saved as '{output_dir}/fish_activity_model.h5'")

# Step 10: Save the training history as a plot
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.savefig(os.path.join(output_dir, 'training_history.png'))  # Save plot as PNG
plt.show()
print(f"Training history plot saved as '{output_dir}/training_history.png'")

In [ ]:
# ===============================================
# Test Fish Activity CNN Model
# ===============================================

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import os
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Paths
test_dir = 'C:\\Users\\STF8586\\OneDrive - University of Derby\\Desktop\\S.K\\EffiDataset\\test'
model_path = 'C:\\Users\\STF8586\\OneDrive - University of Derby\\Desktop\\S.K\\results\\best_model.h5'
output_dir = 'C:\\Users\\STF8586\\OneDrive - University of Derby\\Desktop\\S.K\\results\\test'

# Step 2: Load the trained model
model = tf.keras.models.load_model(model_path)
print(f"Model loaded from {model_path}")

# Step 3: Prepare test data generator
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

# Step 4: Evaluate on test set
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc*100:.2f}%")

# Step 5: Make predictions
test_generator.reset()
pred_probs = model.predict(test_generator)
pred_classes = (pred_probs > 0.5).astype(int).flatten()

# True labels
true_classes = test_generator.classes

# Step 6: Save predictions
np.save(os.path.join(output_dir, 'test_predictions.npy'), pred_classes)
np.save(os.path.join(output_dir, 'test_true_labels.npy'), true_classes)
print(f"Predictions and true labels saved in '{output_dir}' folder!")

# Step 7: Confusion Matrix and Classification Report
cm = confusion_matrix(true_classes, pred_classes)
cr = classification_report(true_classes, pred_classes, target_names=test_generator.class_indices.keys())
print("Classification Report:\n", cr)

# Step 8: Plot Confusion Matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=test_generator.class_indices.keys(), yticklabels=test_generator.class_indices.keys())
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.savefig(os.path.join(output_dir, 'confusion_matrix.png'))
plt.show()
print(f"Confusion matrix saved in '{output_dir}/confusion_matrix.png'")